In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms

from collections import defaultdict

from train_bpe import *

%run ./train_bpe.py

c:\Users\weiwe\anaconda3\envs\bpe\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_size = 1

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [19]:
def reshape_to_tuples(data, dim):
    if isinstance(data, torch.Tensor):
        if data.shape[0] == 1:
            data = data.squeeze().numpy()
    # data = (data * 255).astype(int)
    rows, cols = data.shape
    row_group_size = rows // dim[0]
    col_group_size = cols // dim[1]
    
    row_indices = []
    col_indices = []
    for i in range(0, row_group_size):
        row_indices.append([j for j in range(i, rows, row_group_size)])
        
    for i in range(0, col_group_size):
        col_indices.append([j for j in range(i, cols, col_group_size)])

    # print(row_indices)
    # print(col_indices)

    tuples = []
    indices_matrix = []

    for row_indices_group in row_indices:
        for col_indices_group in col_indices:
            group = []
            indices_tuple = []

            for r_idx in row_indices_group:
                for c_idx in col_indices_group:
                    group.append(data[r_idx, c_idx])
                    indices_tuple.append((r_idx, c_idx))
            
            indices_matrix.append(tuple(indices_tuple))
            tuples.append(tuple(group))
            
    return tuples, indices_matrix

In [14]:
arr = np.arange(16).reshape(4, 4)
print('4x4 image:')
print(arr)
print('2x2 tuples:')
print(reshape_to_tuples(arr, (2, 2)))
print('2x1 tuples:')
print(reshape_to_tuples(arr, (2, 1)))
print('1x2 tuples:')
print(reshape_to_tuples(arr, (1, 2)))

4x4 image:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
2x2 tuples:
[[0, 2], [1, 3]]
[[0, 2], [1, 3]]
([(0, 2, 8, 10), (1, 3, 9, 11), (4, 6, 12, 14), (5, 7, 13, 15)], [((0, 0), (0, 2), (2, 0), (2, 2)), ((0, 1), (0, 3), (2, 1), (2, 3)), ((1, 0), (1, 2), (3, 0), (3, 2)), ((1, 1), (1, 3), (3, 1), (3, 3))])
2x1 tuples:
[[0, 2], [1, 3]]
[[0], [1], [2], [3]]
([(0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5, 13), (6, 14), (7, 15)], [((0, 0), (2, 0)), ((0, 1), (2, 1)), ((0, 2), (2, 2)), ((0, 3), (2, 3)), ((1, 0), (3, 0)), ((1, 1), (3, 1)), ((1, 2), (3, 2)), ((1, 3), (3, 3))])
1x2 tuples:
[[0], [1], [2], [3]]
[[0, 2], [1, 3]]
([(0, 2), (1, 3), (4, 6), (5, 7), (8, 10), (9, 11), (12, 14), (13, 15)], [((0, 0), (0, 2)), ((0, 1), (0, 3)), ((1, 0), (1, 2)), ((1, 1), (1, 3)), ((2, 0), (2, 2)), ((2, 1), (2, 3)), ((3, 0), (3, 2)), ((3, 1), (3, 3))])


In [7]:
def freq_pair(tokens):
    pairs = defaultdict(int)
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i+1])
        pairs[pair] += 1
    return pairs

In [8]:
def max_freq_pair(freq_pairs_dic):
    max_freq = None

    for pair, freq in freq_pairs_dic.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    return best_pair, max_freq

In [9]:
# def merge(image_vec, vocab):
#     new_vocab = {key: value for key, value in vocab.items() if key > 255}
#     for token_id, pair in new_vocab.items():
#         i = 0
#         while i < len(image_vec) - 1:
#             if pair[0] == image_vec[i] and pair[1] == image_vec[i+1]:
#                 image_vec[i] = token_id
#                 image_vec = np.delete(image_vec, [i+1], None)
#             i += 1
#     return image_vec

def merge(image_vec, pair, idx):
    np_ids = np.array(image_vec)
    idx_to_remove = []
    i = 0
    while i < len(np_ids) - 1:
        if pair[0] == np_ids[i] and pair[1] == np_ids[i+1]:
            np_ids[i] = idx
            idx_to_remove.append(i+1)
            i += 1
        i += 1
    np_ids = np.delete(np_ids, idx_to_remove)
    return np_ids

In [10]:
def tokenize(image, size, vocab, min_freq=2):
    if image.ndim != 1:
        image = image.flatten()
    image = image.astype(int)
    
    while len(vocab) < size:
        pair, freq = max_freq_pair(freq_pair(image))

        if freq < min_freq:
            break

        if pair not in vocab.values():
            idx = len(vocab)
            vocab[idx] = pair
        else:
            for key, val in vocab.items():
                if val == pair:
                    idx = key
        
        image = merge(image, pair, idx)
   
    return image, vocab

In [20]:
dim = (4, 4)

for image, label in tqdm(train_loader):
    image = reshape_to_tuples(image, dim)
    break

  0%|          | 0/60000 [00:00<?, ?it/s]
